# Setup thư viện

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.3 MB/s eta 0:00:00


# Predict với pretrain weights




In [2]:
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt

--2026-03-28 02:57:30--  https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/732c503e-9fcb-4a82-be7f-106baafbda15?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-28T03%3A42%3A11Z&rscd=attachment%3B+filename%3Dyolov8n.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-28T02%3A41%3A32Z&ske=2026-03-28T03%3A42%3A11Z&sks=b&skv=2018-11-09&sig=TkpDXdm1DHhsSYmGubHlvuINkezeRw8KOCtPUfWYb%2Bg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NDY2Njk1MSwibmJmIjoxNzc0NjY2NjUxLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmN

In [3]:
!yolo task=detect mode=predict model=yolov8n.pt source="https://media.baobinhphuoc.com.vn/Content/UploadFiles/EditorFiles/images/2017/Quy1/maxresdefault.jpg"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

image 1/1 /content/maxresdefault.jpg: 384x640 1 bird, 78.6ms
Speed: 11.1ms preprocess, 78.6ms inference, 36.8ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [ ]:
# Predict detect bằng Python API
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
# local bật cam
# results = model.predict(show=True, source="0")
results = model.predict(source="https://media.baobinhphuoc.com.vn/Content/UploadFiles/EditorFiles/images/2017/Quy1/maxresdefault.jpg")

# print(results)

In [ ]:
# Download pretrain segmentation
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n-seg.pt

In [ ]:

# Thử predict segmentation
!yolo task=segment mode=predict model=yolov8n-seg.pt source="https://media.baobinhphuoc.com.vn/Content/UploadFiles/EditorFiles/images/2017/Quy1/maxresdefault.jpg"

# Train với custom data

In [ ]:
# Giải nén file data và tổ chức vào thư mục train

# Extract zip data file
%cd /content/drive/MyDrive/data
!unzip /content/drive/MyDrive/data/firedata.zip

# Tổ chức lại các file
%cd /content/drive/MyDrive/data
!mkdir train
!mkdir train/images
!mkdir train/labels
!mv *.jpg train/images
!mv *.txt train/labels

In [ ]:
# Create training yaml file
%cd /content/drive/MyDrive
!rm data/mydataset.yaml # nếu có file thì xóa
!echo 'train: /content/drive/MyDrive/data/train' >> data/mydataset.yaml
!echo 'val: /content/drive/MyDrive/data/train' >> data/mydataset.yaml
!echo 'nc: 1' >> data/mydataset.yaml
!echo "names: ['fire']" >> data/mydataset.yaml

In [ ]:

# Train với CLI
!yolo task=detect mode=train model=yolov8n.pt data=data/mydataset.yaml epochs=1 imgsz=640

In [ ]:
# Train với augmentation built-in
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/drive/MyDrive/data/mydataset.yaml',
    epochs=50,
    imgsz=640,

    #  AUGMENTATION 
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,

    flipud=0.0,
    fliplr=0.5,

    mosaic=1.0,   # Ghép 4 ảnh → 1 ảnh
    mixup=0.2     # trộn 2 ảnh với nhau
)

In [ ]:
# Train với Python API
from ultralytics import YOLO

model = YOLO("yolov8n.pt") # Load pretrain và finetune vài layer cuối. Transfer LEarning

results = model.train(data="data/mydataset.yaml", epochs=1)

In [ ]:
# Detect thử với weights mới train thì làm ntn?
!yolo task=detect mode=predict model="/content/drive/MyDrive/runs/detect/train3/weights/last.pt" source="https://i-vnexpress.vnecdn.net/2019/03/23/chay-1-3737-1553317413.jpg"